# Dark Pattern Detection — Retrain on Combined Dataset
Training DistilRoBERTa on the merged `combined_dataset.csv` (dataset.csv + train.csv, labels aligned).
- **2,839 samples** | Dark: 1,489 | Not Dark: 1,350
- **5 epochs** for better convergence
- Saves to `./distilroberta_darkpattern_detection` (overwrites previous model)

In [3]:
import pandas as pd
import numpy as np
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'http\S+', '', text)
    return " ".join(text.split())

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


Using device: cuda


In [4]:
# Load combined dataset
df = pd.read_csv('combined_dataset.csv')
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() >= 5]
df['label'] = df['label'].astype(int)

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts().to_string()}")

texts = df['text'].tolist()
labels = df['label'].tolist()

# 85/15 split, stratified to preserve class balance
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
print(f"\nTrain: {len(train_texts)} | Val: {len(val_texts)}")

# Tokenize
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_enc   = tokenizer(val_texts,   truncation=True, padding=True, max_length=256)

train_dataset = TextDataset(train_enc, train_labels)
val_dataset   = TextDataset(val_enc,   val_labels)

Total samples: 2839
Label distribution:
label
1    1489
0    1350

Train: 2413 | Val: 426


In [5]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir='./results_combined',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs_combined',
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.313100,0.202279,0.917840,0.917809,0.917869,0.917840
2,0.204700,0.182148,0.934272,0.934256,0.934282,0.934272
3,0.068900,0.322459,0.922535,0.922557,0.922670,0.922535
4,0.080600,0.350797,0.931925,0.931932,0.931949,0.931925
5,0.049200,0.371792,0.927230,0.927222,0.927223,0.927230


TrainOutput(global_step=755, training_loss=0.16152321783122636, metrics={'train_runtime': 632.776, 'train_samples_per_second': 19.067, 'train_steps_per_second': 1.193, 'total_flos': 668005666531080.0, 'train_loss': 0.16152321783122636, 'epoch': 5.0})

In [6]:
# Full evaluation report on validation set
print("\n=== Final Evaluation ===")
preds_output = trainer.predict(val_dataset)
preds = preds_output.predictions.argmax(-1)
print(classification_report(val_labels, preds, target_names=["Not Dark Pattern", "Dark Pattern"]))

# Save model
save_path = "./distilroberta_darkpattern_detection"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"\nModel saved to {save_path}")


=== Final Evaluation ===


                  precision    recall  f1-score   support

Not Dark Pattern       0.93      0.92      0.92       203
    Dark Pattern       0.93      0.93      0.93       223

        accuracy                           0.93       426
       macro avg       0.93      0.93      0.93       426
    weighted avg       0.93      0.93      0.93       426


Model saved to ./distilroberta_darkpattern_detection


## Sanity Check on Test Strings

In [7]:
# Quick sanity check
test_texts = [
    "Only 2 items left in stock! Order NOW before it's gone!",
    "Pillowcases & Shams set in blue",
    "1,234 people are viewing this item right now",
    "Satisfaction guaranteed or your money back",
    "Add insurance for only $1.99/month - pre-selected for you",
    "Write a review",
    "FREE SHIPPING - today only!",
    "Blue denim jacket, regular fit",
    "No thanks, I'll pay full price and miss out",
    "Subscribe Now"
]

model.eval()
print(f"{'TEXT':<52} {'LABEL':<18} {'CONF':>5}")
print("-" * 77)
with torch.no_grad():
    for text in test_texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           padding=True, max_length=256).to(device)
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze()
        label = torch.argmax(probs).item()
        conf = probs[label].item()
        label_name = "Dark Pattern" if label == 1 else "Not Dark Pattern"
        print(f"{text[:51]:<52} {label_name:<18} {conf:>5.1%}")

TEXT                                                 LABEL               CONF
-----------------------------------------------------------------------------
Only 2 items left in stock! Order NOW before it's g  Dark Pattern       100.0%
Pillowcases & Shams set in blue                      Not Dark Pattern   100.0%
1,234 people are viewing this item right now         Dark Pattern       100.0%
Satisfaction guaranteed or your money back           Not Dark Pattern   99.8%
Add insurance for only $1.99/month - pre-selected f  Dark Pattern       100.0%
Write a review                                       Not Dark Pattern   100.0%
FREE SHIPPING - today only!                          Dark Pattern       100.0%
Blue denim jacket, regular fit                       Not Dark Pattern   100.0%
No thanks, I'll pay full price and miss out          Dark Pattern       100.0%
Subscribe Now                                        Not Dark Pattern   100.0%
